# 3.2.4 — Projeção combinada de **Sentenças + Documentos** com BERT

**Aluna:** Bruna Soto Cardoso dos Santos  
**Corpus:** 35 documentos — Ciência e Tecnologia no Brasil (AD1)  
**Repositório:** https://github.com/brusoto/SRI_AD_TE

---
Projeta **sentenças e documentos no mesmo espaço vetorial** BERT.  
Cada sentença usa mean-pooling dos tokens; cada documento usa o token [CLS].

**Saídas** (salvas em `projecao/`):
- `tensors_sentenca_documento.tsv`
- `metadata_sentenca_documento.tsv`
- `config_sentenca_documento.json`

## Célula 1 — Instalação

In [ ]:
!pip install transformers torch spacy --quiet
!python -m spacy download pt_core_news_sm --quiet

## Célula 2 — Drive e caminhos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/SRI'
DATA_PATH = os.path.join(BASE_PATH, 'data')
PROJ_PATH = os.path.join(BASE_PATH, 'projecao')
os.makedirs(PROJ_PATH, exist_ok=True)
CSV_FILE  = os.path.join(DATA_PATH, 'documentos.csv')
print(f'CSV: {CSV_FILE}')

## Célula 3 — Baixar corpus se necessário

In [ ]:
if not os.path.exists(CSV_FILE):
    os.makedirs(DATA_PATH, exist_ok=True)
    URL = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/data/documentos.csv'
    !wget -q "{URL}" -O "{CSV_FILE}"
    print('Baixado.')
else:
    print('Arquivo já existe.')

## Célula 4 — Imports e corpus

In [ ]:
import pandas as pd
import torch
import numpy as np
import spacy
from transformers import BertTokenizer, BertModel

# spaCy para segmentação de sentenças
nlp = spacy.load('pt_core_news_sm')

df = pd.read_csv(CSV_FILE)
df.columns = [c.strip().lower() for c in df.columns]
if 'id' not in df.columns:
    df.insert(0, 'id', range(1, len(df)+1))

print(f'Documentos: {len(df)}')
df.head(2)

## Célula 5 — Modelo BERT

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)
model      = BertModel.from_pretrained(MODEL_NAME)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f'Dispositivo: {device}')

## Célula 6 — Funções de embedding

In [ ]:
def get_document_embedding(text):
    """Embedding do documento via token [CLS]."""
    inputs = tokenizer(
        text, return_tensors='pt',
        max_length=512, truncation=True, padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze().cpu().numpy()


def get_sentence_embedding(sentence):
    """Embedding da sentença via mean-pooling dos tokens (exclui [CLS]/[SEP]/[PAD])."""
    inputs = tokenizer(
        sentence, return_tensors='pt',
        max_length=128, truncation=True, padding='max_length'
    )
    input_ids      = inputs['input_ids']
    attention_mask = inputs['attention_mask']
    inputs_dev     = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs_dev)

    hidden = out.last_hidden_state.squeeze(0).cpu()  # (128, 768)
    mask   = attention_mask.squeeze(0).unsqueeze(-1).float()  # (128, 1)

    # Zera [CLS] (idx 0) e [SEP] na última posição real
    mask[0] = 0
    seq_len = int(attention_mask.sum()) - 1
    if seq_len > 0:
        mask[seq_len] = 0

    # Mean pooling
    sum_hidden  = (hidden * mask).sum(dim=0)
    count       = mask.sum()
    mean_hidden = (sum_hidden / count).numpy()
    return mean_hidden

## Célula 7 — Geração dos embeddings (sentenças + documentos)

In [ ]:
all_embeddings = []
all_metadata   = []  # (label, tipo, doc_id, sentenca_idx)

for _, row in df.iterrows():
    doc_id = row['id']
    texto  = str(row['documento'])

    # ── 1. Embedding do DOCUMENTO ─────────────────────────────────────────
    doc_emb = get_document_embedding(texto)
    resumo  = ' '.join(texto.split()[:5])
    all_embeddings.append(doc_emb)
    all_metadata.append((f'DOC_{doc_id}: {resumo}', 'DOCUMENTO', f'Doc_{doc_id}', 0))

    # ── 2. Segmenta em sentenças com spaCy ───────────────────────────────
    doc_spacy = nlp(texto)
    sentencas = [s.text.strip() for s in doc_spacy.sents if len(s.text.strip()) > 10]

    for s_idx, sent in enumerate(sentencas):
        sent_emb = get_sentence_embedding(sent)
        # Primeiros 40 chars da sentença como label
        sent_label = sent[:40].replace('\n', ' ')
        all_embeddings.append(sent_emb)
        all_metadata.append((
            f'S{s_idx+1}[Doc_{doc_id}]: {sent_label}',
            'SENTENCA',
            f'Doc_{doc_id}',
            s_idx + 1
        ))

    print(f'Doc_{doc_id:02d}: 1 doc + {len(sentencas)} sentenças')

all_embeddings_np = np.array(all_embeddings)
print(f'\nTotal pontos: {len(all_metadata)} | Shape: {all_embeddings_np.shape}')

## Célula 8 — Salvar arquivos TSV

In [ ]:
# tensors
tensor_path = os.path.join(PROJ_PATH, 'tensors_sentenca_documento.tsv')
np.savetxt(tensor_path, all_embeddings_np, delimiter='\t')
print(f'Salvo: {tensor_path}')

# metadata
meta_path = os.path.join(PROJ_PATH, 'metadata_sentenca_documento.tsv')
with open(meta_path, 'w', encoding='utf-8') as f:
    f.write('label\ttipo\tdoc\tsentenca_idx\n')
    for label, tipo, doc, sidx in all_metadata:
        f.write(f'{label}\t{tipo}\t{doc}\t{sidx}\n')
print(f'Salvo: {meta_path}')

## Célula 9 — Gerar config_sentenca_documento.json

In [ ]:
import json

GITHUB_RAW = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao'
config = {
    "embeddings": [
        {
            "tensorName": "Sentenças e Documentos — Ciência e Tecnologia BR",
            "tensorShape": [len(all_metadata), 768],
            "tensorPath": f"{GITHUB_RAW}/tensors_sentenca_documento.tsv",
            "metadataPath": f"{GITHUB_RAW}/metadata_sentenca_documento.tsv"
        }
    ]
}

config_path = os.path.join(PROJ_PATH, 'config_sentenca_documento.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f'Salvo: {config_path}')

## Célula 10 — Visualização PCA com distinção SENTENÇA vs DOCUMENTO

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(all_embeddings_np)

n_docs = len(df)
cmap   = cm.get_cmap('tab20', n_docs)

fig, ax = plt.subplots(figsize=(16, 11))

for i, (x, y) in enumerate(coords):
    label, tipo, doc, sidx = all_metadata[i]
    doc_num = int(doc.replace('Doc_', '')) - 1
    color   = cmap(doc_num)

    if tipo == 'DOCUMENTO':
        ax.scatter(x, y, color='red', s=150, marker='*', zorder=5,
                   edgecolors='darkred', linewidths=0.8)
        ax.annotate(
            doc.replace('Doc_', 'D'),
            (x, y), fontsize=7, fontweight='bold', color='darkred',
            xytext=(4, 4), textcoords='offset points'
        )
    else:  # SENTENCA
        ax.scatter(x, y, color=color, s=30, alpha=0.65, zorder=3,
                   marker='o', edgecolors='none')
        if sidx == 1:  # anota apenas a 1ª sentença de cada doc
            ax.annotate(
                f'S1/{doc.replace("Doc_","D")}',
                (x, y), fontsize=6, alpha=0.75, color=color,
                xytext=(2, 2), textcoords='offset points'
            )

legend = [
    mpatches.Patch(color='red',       label='Documentos (embedding [CLS])'),
    mpatches.Patch(color='steelblue', label='Sentenças (mean-pooling BERT)'),
]
ax.legend(handles=legend, loc='upper right', fontsize=9)
ax.set_title(
    'Projeção PCA — Sentenças e Documentos no mesmo espaço (BERT)\n'
    'Corpus: Ciência e Tecnologia no Brasil',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(PROJ_PATH, 'grafico_sentenca_documento.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {fig_path}')

## Célula 11 — Análise: sentenças mais próximas do documento

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Escolha um documento para analisar
DOC_ANALISE = 1  # ← altere para o ID do documento que deseja analisar

# Índices do documento e suas sentenças
doc_indices  = [i for i, m in enumerate(all_metadata)
                if m[2] == f'Doc_{DOC_ANALISE}' and m[1] == 'DOCUMENTO']
sent_indices = [i for i, m in enumerate(all_metadata)
                if m[2] == f'Doc_{DOC_ANALISE}' and m[1] == 'SENTENCA']

if doc_indices and sent_indices:
    doc_emb_vec  = all_embeddings_np[doc_indices[0]].reshape(1, -1)
    sent_embs    = all_embeddings_np[sent_indices]
    sims         = cosine_similarity(doc_emb_vec, sent_embs)[0]

    print(f'Similaridade coseno — Doc_{DOC_ANALISE} vs suas sentenças:')
    print('-' * 60)
    for rank, idx in enumerate(np.argsort(sims)[::-1]):
        m = all_metadata[sent_indices[idx]]
        print(f'  #{rank+1} sim={sims[idx]:.4f} | {m[0][:60]}')
else:
    print(f'Doc_{DOC_ANALISE} não encontrado.')

## Célula 12 — Link para o TensorFlow Embedding Projector

In [ ]:
PROJECTOR_URL = (
    'https://projector.tensorflow.org/?config='
    'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao/config_sentenca_documento.json'
)
print('═' * 70)
print('LINK DO EMBEDDING PROJECTOR — SENTENÇAS + DOCUMENTOS:')
print(PROJECTOR_URL)
print('═' * 70)
print('\nArquivos gerados em projecao/:')
for fname in sorted(os.listdir(PROJ_PATH)):
    fpath = os.path.join(PROJ_PATH, fname)
    size  = os.path.getsize(fpath)
    print(f'  {fname:50s} {size/1024:.1f} KB')